# 07 — TMDB Keyword Valence

Matches all 84K TMDB keywords against the joined SCL lexicon (from `06_scl_join`)
using the OPP → NMA → VAD hierarchy, then computes per-keyword and aggregate valence features.

## Matching strategy

```
1. Exact phrase match against joined lexicon (OPP > NMA > VAD already resolved into `valence`)
2. Token-mean fallback: average the hierarchy-selected `valence` of each unigram token
```

The token-mean fallback uses `valence` (not raw `vad_valence`), so if a token has an
OPP or NMA score it takes precedence even in the fallback.

## Output columns (per keyword)

| column | description |
|--------|-------------|
| `keyword` | TMDB keyword name |
| `ngrams` | word count |
| `valence` | best available valence score, bipolar [-1, +1] |
| `valence_source` | `opp` / `nma` / `vad` / `token_mean` |
| `valence_count` | number of lexicons with a score (from joined; 0 if token_mean) |
| `valence_variance` | variance across lexicons (null if < 2 sources) |
| `matched_tokens` | tokens with a valence score (for token_mean fallback) |
| `total_tokens` | total tokens in keyword |

## Aggregate features (per movie's keyword set)

```
matched_keyword_count   — keywords with any valence score
avg_keyword_valence     — mean valence across matched keywords
min_keyword_valence     — most negative keyword
max_keyword_valence     — most positive keyword
valence_std             — spread of sentiment across keywords
```

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

DATA = Path('data')

## Load joined lexicon and TMDB keywords

In [2]:
lexicon = pd.read_pickle(DATA / 'lexicons/scl_joined.pkl')

# deduplicate: keep first occurrence per term
# (valence_source priority OPP > NMA > VAD is already baked into `valence`)
lex_dedup = (
    lexicon[['term', 'valence', 'valence_source', 'valence_count', 'valence_variance']]
    .dropna(subset=['term'])
    .drop_duplicates(subset='term', keep='first')
)
print(f'Lexicon after dedup: {len(lex_dedup):,} unique terms (from {len(lexicon):,})')

# fast phrase lookup
lex_lookup = lex_dedup.set_index('term').to_dict('index')

# unigram lookup using hierarchy-selected `valence` (not raw vad_valence)
# so OPP/NMA scores take precedence even in token-mean fallback
unigram_lookup = (
    lex_dedup[lex_dedup['term'].str.split().str.len() == 1]
    .set_index('term')['valence']
    .dropna()
    .to_dict()
)

print(f'Phrase lookup entries: {len(lex_lookup):,}')
print(f'Unigram entries for token-mean fallback: {len(unigram_lookup):,}')

Lexicon after dedup: 57,001 unique terms (from 57,005)
Phrase lookup entries: 57,001
Unigram entries for token-mean fallback: 44,811


In [3]:
keywords = pd.read_csv(DATA / 'tmdb_keywords_canonical.csv')
print(f'TMDB keywords: {len(keywords):,}')
keywords.head(5)

TMDB keywords: 84,765


,tmdb_keyword_id,name
0,30,individual
1,65,holiday
2,74,germany
3,75,gunslinger
4,83,saving the world


## Score each keyword

In [4]:
def score_keyword(name):
    key = str(name).lower().strip()
    tokens = key.split()
    n = len(tokens)

    # 1. exact phrase match — valence already hierarchy-resolved (OPP > NMA > VAD)
    if key in lex_lookup:
        row = lex_lookup[key]
        return {
            'valence':          row['valence'],
            'valence_source':   row['valence_source'],
            'valence_count':    row['valence_count'],
            'valence_variance': row['valence_variance'],
            'matched_tokens':   n,
            'total_tokens':     n,
        }

    # 2. token-mean fallback using hierarchy-selected unigram valence
    token_scores = [unigram_lookup[t] for t in tokens if t in unigram_lookup]
    if token_scores:
        return {
            'valence':          float(np.mean(token_scores)),
            'valence_source':   'token_mean',
            'valence_count':    0,
            'valence_variance': float(np.var(token_scores, ddof=1)) if len(token_scores) > 1 else None,
            'matched_tokens':   len(token_scores),
            'total_tokens':     n,
        }

    # 3. no match
    return {
        'valence':          None,
        'valence_source':   None,
        'valence_count':    0,
        'valence_variance': None,
        'matched_tokens':   0,
        'total_tokens':     n,
    }

scores = keywords['name'].apply(score_keyword).apply(pd.Series)
kw_scored = pd.concat([keywords, scores], axis=1)
kw_scored['ngrams'] = kw_scored['name'].str.split().str.len()

kw_scored = kw_scored[[
    'tmdb_keyword_id', 'name', 'ngrams',
    'valence', 'valence_source', 'valence_count', 'valence_variance',
    'matched_tokens', 'total_tokens',
]]

print(f'Scored: {len(kw_scored):,} keywords')
print(f'\nvalence_source breakdown:')
print(kw_scored['valence_source'].value_counts(dropna=False).to_string())
print(f'\nCoverage: {kw_scored["valence"].notna().sum():,} / {len(kw_scored):,} ({kw_scored["valence"].notna().mean()*100:.1f}%)')
kw_scored.head(5)

Scored: 84,765 keywords

valence_source breakdown:
valence_source
token_mean    36425
NaN           35680
vad           11523
nma             711
opp             426

Coverage: 49,085 / 84,765 (57.9%)


,tmdb_keyword_id,name,ngrams,valence,valence_source,valence_count,valence_variance,matched_tokens,total_tokens
0,30,individual,1.0,0.254,vad,1.0,NaN,1.0,1.0
1,65,holiday,1.0,0.656,opp,2.0,0.028800,1.0,1.0
2,74,germany,1.0,NaN,NaN,0.0,NaN,0.0,1.0
3,75,gunslinger,1.0,-0.368,vad,1.0,NaN,1.0,1.0
4,83,saving the world,3.0,0.327,token_mean,0.0,0.000002,2.0,3.0


## Valence distribution

In [5]:
matched = kw_scored[kw_scored['valence'].notna()]

print('Valence stats (matched keywords only):')
print(matched['valence'].describe().round(4).to_string())

print(f'\nPositive (>0):  {(matched["valence"] > 0).sum():,}  ({(matched["valence"] > 0).mean()*100:.1f}%)')
print(f'Neutral  (=0):  {(matched["valence"] == 0).sum():,}  ({(matched["valence"] == 0).mean()*100:.1f}%)')
print(f'Negative (<0):  {(matched["valence"] < 0).sum():,}  ({(matched["valence"] < 0).mean()*100:.1f}%)')

print(f'\nMost positive keywords:')
print(matched.nlargest(10, 'valence')[['name', 'valence', 'valence_source']].to_string(index=False))

print(f'\nMost negative keywords:')
print(matched.nsmallest(10, 'valence')[['name', 'valence', 'valence_source']].to_string(index=False))

Valence stats (matched keywords only):
count    49085.0000
mean         0.0951
std          0.3829
min         -1.0000
25%         -0.1255
50%          0.1290
75%          0.3600
max          1.0000

Positive (>0):  30,882  (62.9%)
Neutral  (=0):  1,279  (2.6%)
Negative (<0):  16,924  (34.5%)

Most positive keywords:
           name  valence valence_source
        sapient      1.0            vad
    empowerment      1.0            vad
       pacifism      1.0            vad
suburbian idyll      1.0     token_mean
   mercifulness      1.0            vad
   thankfulness      1.0            vad
      precocity      1.0            vad
         tahiti      1.0            vad
 mahatma gandhi      1.0     token_mean
invulnerability      1.0            vad

Most negative keywords:
            name  valence valence_source
    ku klux klan     -1.0     token_mean
     nuclear war     -1.0            vad
       cataclysm     -1.0            vad
 excommunication     -1.0            vad
         bu

## Aggregate features

Per movie: average valence, spread, and count across its keyword set.
Demonstrated on a simulated example keyword set.

In [6]:
print("""
For a movie M with keywords K = {k1, k2, ..., kn} where valence(ki) is defined:

  matched_keyword_count = |{ki : valence(ki) is not null}|
  avg_keyword_valence   = mean(valence(ki) for matched ki)
  min_keyword_valence   = min(valence(ki) for matched ki)
  max_keyword_valence   = max(valence(ki) for matched ki)
  valence_std           = std(valence(ki) for matched ki, ddof=1)

valence_source precedence: opp > nma > vad > token_mean
""")

example_keywords = ['dark comedy', 'tragic love', 'happy ending', 'serial killer', 'redemption']
example_rows = []
for kw in example_keywords:
    s = score_keyword(kw)
    example_rows.append({'keyword': kw, **s})

ex = pd.DataFrame(example_rows)
matched_ex = ex[ex['valence'].notna()]

print('Example keyword set:')
print(ex[['keyword', 'valence', 'valence_source']].to_string(index=False))
print(f'\nmatched_keyword_count : {len(matched_ex)}')
print(f'avg_keyword_valence   : {matched_ex["valence"].mean():.4f}')
print(f'min_keyword_valence   : {matched_ex["valence"].min():.4f}')
print(f'max_keyword_valence   : {matched_ex["valence"].max():.4f}')
print(f'valence_std           : {matched_ex["valence"].std(ddof=1):.4f}')


For a movie M with keywords K = {k1, k2, ..., kn} where valence(ki) is defined:

  matched_keyword_count = |{ki : valence(ki) is not null}|
  avg_keyword_valence   = mean(valence(ki) for matched ki)
  min_keyword_valence   = min(valence(ki) for matched ki)
  max_keyword_valence   = max(valence(ki) for matched ki)
  valence_std           = std(valence(ki) for matched ki, ddof=1)

valence_source precedence: opp > nma > vad > token_mean

Example keyword set:
      keyword  valence valence_source
  dark comedy  -0.1080            vad
  tragic love  -0.0085     token_mean
 happy ending   0.9000            vad
serial killer  -0.6420            vad
   redemption  -0.2480            vad

matched_keyword_count : 5
avg_keyword_valence   : -0.0213
min_keyword_valence   : -0.6420
max_keyword_valence   : 0.9000
valence_std           : 0.5686


## Save

In [7]:
out = DATA / 'lexicons/tmdb_keyword_valence.pkl'
kw_scored.to_pickle(out)
print(f'Saved → {out}  ({out.stat().st_size / 1e6:.1f} MB)')
print(f'Shape: {kw_scored.shape}')

Saved → data/lexicons/tmdb_keyword_valence.pkl  (6.6 MB)
Shape: (84765, 9)
